### **Fase 0**: Import Generico

In [13]:
import os
import sys
import importlib
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive già montato, salto il mount.")

sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
importlib.invalidate_caches()

import common
importlib.reload(common)
from common import *

print(f"Device attivo: {device}")
print(paths)
log('Notebook avviato correttamente')

Drive già montato, salto il mount.
Device attivo: cuda
{'raw_images': '/content/drive/MyDrive/progetto_immunogold/raw_images', 'annotations': '/content/drive/MyDrive/progetto_immunogold/annotations', 'patches_images': '/content/drive/MyDrive/progetto_immunogold/patches/images', 'patches_coordinates': '/content/drive/MyDrive/progetto_immunogold/patches/coordinates', 'patches_density': '/content/drive/MyDrive/progetto_immunogold/patches/density_maps', 'checkpoints': '/content/drive/MyDrive/progetto_immunogold/checkpoints', 'logs': '/content/drive/MyDrive/progetto_immunogold/logs'}
[2026-09-29 15:13] Notebook avviato correttamente


### **Import modello**

Richiamiamo il file modello in cui abbiamo incluso tutta la costruzione della rete (encoder-decoder).

In [2]:
sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
from modello import UNetImmunogold

### **Dataset PyTorch**

Viene definita la classe che richiama insieme patch e density map associata per il training. Riorganizza le informazioni che abbiamo già costruito (*mappa_patch.csv*, *split_immagini.csv*, ecc.) e le adatta all'utilizzo nella libreria PyTorch.

In [4]:
FATTORE_SCALA = 100.0

from torch.utils.data import Dataset, DataLoader

class DatasetImmunogold(Dataset):
    def __init__(self, split, transform=None):
        split_df = pd.read_csv(os.path.join(paths['annotations'], 'split_immagini.csv'))
        mappa_patch = pd.read_csv(os.path.join(paths['annotations'], 'mappa_patch.csv'))

        immagini_split = split_df[split_df['split'] == split]['id_immagine']
        patch_split = mappa_patch[mappa_patch['id_immagine'].isin(immagini_split)].reset_index(drop=True)

        patch_split['density_map_path'] = patch_split['patch_path'].apply(self._path_density_map)
        patch_split = patch_split[patch_split['density_map_path'].apply(os.path.exists)].reset_index(drop=True)

        self.patches = patch_split
        self.transform = transform

        print(f"Dataset '{split}': {len(self.patches)} patch con density map disponibile")

    @staticmethod
    def _path_density_map(patch_path):
        nome_patch = Path(patch_path).stem
        return os.path.join(paths['patches_density'], f"{nome_patch}.npy")

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        riga = self.patches.iloc[idx]

        with Image.open(riga['patch_path']) as img:
            arr = np.array(img)
        if arr.ndim == 3:
            arr = arr.mean(axis=-1)
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

        density_map = np.load(riga['density_map_path']).astype(np.float32) * FATTORE_SCALA

        immagine_tensor = torch.from_numpy(arr).unsqueeze(0)          # (1, H, W)
        density_tensor = torch.from_numpy(density_map).unsqueeze(0)   # (1, H, W)

        if self.transform is not None:
            immagine_tensor, density_tensor = self.transform(immagine_tensor, density_tensor)

        return immagine_tensor, density_tensor

### **Data Augmentation**

Applicazione di piccole trasformazioni alle immagini dello split di training (mai validation/test), utile per rafforzare la stabilità del nostro modello considerando le poche immagini disponibili per il training (circa 10 al momento).

Per l'esattezza richiamiamo 3 possibili modifiche:

1.   Filp orizzontale/flip verticale: ribaltamento dell'array verticalmente o orizzontalmente (no perdita/approssimazione)
2.   Rotazione: dato un determinato angolo, si ruotano immagine e density map
3.   Contrasto: variazione di contrasto applicabile solamente all'immagine (sulla density map non avrebbe senso, una moltiplicazione per un valore andrebbe solo ad alterare la somma finale)

*augmentazione_train* viene applicata a ogni singola patch del batch. Ripetendo il training su tutte le patch dei vari batch per più epoche (generando dunque versioni ogni volta diverse delle varie patch, grazie a *random*, diamo una maggiore variabilità alle immagini che usiamo per addestrare la rete)

In [5]:
import random
import torchvision.transforms.functional as TF

def augmentazione_train(immagine, density, p_flip=0.5, angolo_max=15, p_rotazione=0.5, p_contrasto=0.5):
    # Flip orizzontale
    if random.random() < p_flip:
        immagine = torch.flip(immagine, dims=[-1])
        density = torch.flip(density, dims=[-1])

    # Flip verticale
    if random.random() < p_flip:
        immagine = torch.flip(immagine, dims=[-2])
        density = torch.flip(density, dims=[-2])

    # Piccola rotazione
    if random.random() < p_rotazione:
        angolo = random.uniform(-angolo_max, angolo_max)
        somma_prima = density.sum()

        immagine = TF.rotate(immagine, angolo, interpolation=TF.InterpolationMode.BILINEAR)
        density = TF.rotate(density, angolo, interpolation=TF.InterpolationMode.BILINEAR)

        somma_dopo = density.sum()
        if somma_dopo > 0:
            density = density * (somma_prima / somma_dopo)  # ripristina la somma esatta persa nell'interpolazione

    # Variazione di contrasto, SOLO sull'immagine
    if random.random() < p_contrasto:
        fattore = random.uniform(0.8, 1.2)
        media = immagine.mean()
        immagine = (immagine - media) * fattore + media
        immagine = torch.clamp(immagine, 0.0, 1.0)

    return immagine, density

dataset_train = DatasetImmunogold(split='train', transform=augmentazione_train)
dataset_val = DatasetImmunogold(split='val')  # nessuna augmentation

Dataset 'train': 793 patch con density map disponibile
Dataset 'val': 315 patch con density map disponibile


### **DataLoader**

Con questa funzione andiamo a definire la dimensione del batch (*BATCH_SIZE*) (parametro importante anche in futuro per il tuning).

*shuffle = True* per il train così che a ogni epoca la composizione dei vari batch cambia (le patch presenti nei batch cambiano), e per evitare che la rete bilanci i pesi seguendo magari dei pattern dovuti all'ordine con cui vengono processate le patch.

*shuffle = False* per il validation siccome qui dobbiamo solamente valutare le metriche e non tarare i pesi, quindi non è importante l'ordine.

In [6]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)

### **Loss e Optimizer**


*   *nn.MSELoss* calcola per ogni pixel (predetto-target)^2 e poi fa la media su tutti i pixel
*   *torch.optim.Adam* è l'algoritmo che aggiorna il singolo peso dopo aver calcolato la loss.
*   *lr* (learning rate) è la dimensione del singolo passo di correzione. Se è troppo grande si rischia di arrivare a una situazione di oscillazione, mentre se è troppo piccolo il training sarà lentissimo a migliorare.
*   *modello_parameters* dice all'optimizer su quali parametri può lavorare.

**Come lavorano su un singolo passo di allenamento?**


1.   Si fornisce un batch di patch al modello che produce le sue density map predette.
2.   Si confronta la predizione con il target vero (le annotazioni), calcolando il *criterio_loss* ovvero un valore che ci dice quanto la rete ha sbagliato sul batch specifico.
3.   Viene richiamato *Backward_pass*, che calcola su ogni singolo parametro quanto questo ha contribuito sull'errore appena misurato. Questo viene calcolato facendo la derivata parziale della loss rispetto a ogni peso.
4.   L'optimizer applica la correzione aggiornando i pesi nella direzione associata all'errore (gradiente) cercando di ridurre la loss al prossimo giro. *lr* controlla la dimensione di questo spostamento.
5. optimizer.zero_grad() è essenziale siccome i gradienti calcolati con backward tendono a sommarsi, quindi bisogna azzerarli a ogni passaggio del batch successivo.






In [7]:
criterio_loss = nn.MSELoss()

modello = UNetImmunogold(canali_in=1, canali_base=16).to(device)
optimizer = torch.optim.Adam(modello.parameters(), lr=1e-3)

### **Training**

In [8]:
NUM_EPOCHS = 5

storico_train_loss = []
storico_val_loss = []
migliore_val_loss = float('inf')

path_checkpoint = os.path.join(paths['checkpoints'], 'miglior_modello.pth')

for epoca in range(1, NUM_EPOCHS + 1):
    modello.train()
    loss_train_epoca = 0.0

    for batch_immagini, batch_density in train_loader:
        batch_immagini = batch_immagini.to(device)
        batch_density = batch_density.to(device)

        optimizer.zero_grad()
        predizione = modello(batch_immagini)
        loss = criterio_loss(predizione, batch_density)
        loss.backward()
        optimizer.step()

        loss_train_epoca += loss.item() * batch_immagini.size(0)

    loss_train_epoca /= len(dataset_train)

    modello.eval()
    loss_val_epoca = 0.0
    with torch.no_grad():
        for batch_immagini, batch_density in val_loader:
            batch_immagini = batch_immagini.to(device)
            batch_density = batch_density.to(device)

            predizione = modello(batch_immagini)
            loss = criterio_loss(predizione, batch_density)
            loss_val_epoca += loss.item() * batch_immagini.size(0)

    loss_val_epoca /= len(dataset_val)

    storico_train_loss.append(loss_train_epoca)
    storico_val_loss.append(loss_val_epoca)

    if loss_val_epoca < migliore_val_loss:
        migliore_val_loss = loss_val_epoca
        torch.save(modello.state_dict(), path_checkpoint)
        marcatore = " <- salvato (migliore finora)"
    else:
        marcatore = ""

    print(f"Epoca {epoca:3d}/{NUM_EPOCHS} - train loss: {loss_train_epoca:.5f} - val loss: {loss_val_epoca:.5f}{marcatore}")
    log(f"Epoca {epoca} - train_loss={loss_train_epoca:.5f} val_loss={loss_val_epoca:.5f}")

Epoca   1/5 - train loss: 0.02651 - val loss: 0.00735 <- salvato (migliore finora)
[2026-09-29 15:08] Epoca 1 - train_loss=0.02651 val_loss=0.00735
Epoca   2/5 - train loss: 0.00239 - val loss: 0.00735
[2026-09-29 15:09] Epoca 2 - train_loss=0.00239 val_loss=0.00735
Epoca   3/5 - train loss: 0.00241 - val loss: 0.00735
[2026-09-29 15:09] Epoca 3 - train_loss=0.00241 val_loss=0.00735
Epoca   4/5 - train loss: 0.00240 - val loss: 0.00735
[2026-09-29 15:09] Epoca 4 - train_loss=0.00240 val_loss=0.00735
Epoca   5/5 - train loss: 0.00239 - val loss: 0.00735
[2026-09-29 15:09] Epoca 5 - train_loss=0.00239 val_loss=0.00735


### **Test 1: Andamento della Loss (MSE)**

La linea tratteggiata rappresenta l'andamento della Loss nel caso in cui il modello predice sempre 0, potrebbe accadere siccome si tratta di valori molto piccoli.

In [14]:
# loss di riferimento: la val loss che avrebbe un modello che predice sempre zero
loss_zero = 0.0
with torch.no_grad():
    for _, batch_density in val_loader:
        loss_zero += (batch_density ** 2).mean().item() * batch_density.size(0)
loss_zero /= len(dataset_val)

epoche = range(1, len(storico_train_loss) + 1)
epoca_migliore = int(np.argmin(storico_val_loss)) + 1

plt.figure(figsize=(8, 4))
plt.plot(epoche, storico_train_loss, marker='o', label='train loss')
plt.plot(epoche, storico_val_loss, marker='o', label='val loss')
plt.axhline(loss_zero, color='gray', linestyle='--', label='predire sempre zero')
plt.axvline(epoca_migliore, color='green', linestyle=':', label=f'miglior epoca ({epoca_migliore})')
plt.xlabel('Epoca')
plt.ylabel('MSE')
plt.title('Andamento della loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

TypeError: 'module' object is not callable

### **Test 2: Feedback visivo**

Colonne: PATCH

Riga 1: Immagine Patch originale;

Riga 2: Density Map annotata;

Riga 3: Density Map predetta.

Con questo comando possiamo visualizzare su delle patch casuali come si è comportato il modello rispetto alle annotazioni originali.

In [15]:
import random

N_ESEMPI = 4
SOLO_CON_PUNTINI = True  # False per includere anche patch di solo sfondo

# carica il MIGLIOR modello salvato, non l'ultimo in memoria
modello.load_state_dict(torch.load(path_checkpoint, map_location=device))
modello.eval()

def scegli_indici_casuali(dataset, n, solo_con_puntini=True):
    indici = list(range(len(dataset)))
    random.shuffle(indici)
    scelti = []
    for i in indici:
        _, density = dataset[i]
        if not solo_con_puntini or density.sum().item() > 0.1:
            scelti.append(i)
        if len(scelti) == n:
            break
    return scelti

indici = scegli_indici_casuali(dataset_val, N_ESEMPI, SOLO_CON_PUNTINI)

fig, axes = plt.subplots(3, len(indici), figsize=(4 * len(indici), 12), squeeze=False)

for col, i in enumerate(indici):
    immagine, density = dataset_val[i]
    with torch.no_grad():
        predizione = modello(immagine.unsqueeze(0).to(device)).squeeze().cpu().numpy()
    density_np = density.squeeze().numpy()
    vmax = max(density_np.max(), predizione.max())  # stessa scala colori per target e predizione

    axes[0, col].imshow(immagine.squeeze().numpy(), cmap='gray')
    axes[0, col].set_title(f"immagine (val #{i})", fontsize=9)
    axes[1, col].imshow(density_np, cmap='magma', vmin=0, vmax=vmax)
    axes[1, col].set_title(f"target: somma {density_np.sum() / FATTORE_SCALA:.1f}", fontsize=9)
    axes[2, col].imshow(predizione, cmap='magma', vmin=0, vmax=vmax)
    axes[2, col].set_title(f"predetto: somma {predizione.sum() / FATTORE_SCALA:.1f}", fontsize=9)
    for r in range(3):
        axes[r, col].axis('off')

plt.tight_layout()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'subplots'

In [ ]:
stat = []
for i in range(0, len(dataset_val), 5):
    _, d = dataset_val[i]
    stat.append((d.max().item(), (d ** 2).mean().item()))
stat = np.array(stat)
print(f"picco medio del target: {stat[:, 0].mean():.5f}")
print(f"MSE predicendo sempre zero: {stat[:, 1].mean():.2e}")

picco medio del target: 0.00556
MSE predicendo sempre zero: 7.67e-07
